In [ ]:
"""
OpenAlex citation diffusion pipeline (single DOI):
1) Get target work by DOI
2) Fetch all citing works (forward citations) via filter=cites:<target_openalex_id>
3) Extract citing-paper countries (paper-level unique countries)
4) Extract institution geo points and count citing papers per institution
5) Save CSV outputs
6) Generate 3 figures:
   - World choropleth map (country-level)
   - Country bar chart (top N)
   - Institution-level pin map (scatter geo)

Outputs:
- citing_works.csv
- country_counts.csv
- nodes.csv / edges.csv (for Gephi)
- institution_geo_counts.csv
- citation_country_map.html/.png
- citation_country_bar.html/.png
- citation_institution_pin_map.html/.png
"""

import time
import requests
import pandas as pd
from collections import Counter, defaultdict

import plotly.express as px
import pycountry


# =========================
# CONFIG
# =========================
TARGET_DOI = "10.1016/j.trd.2021.103159"  # <-- change to your DOI (no https://doi.org/)
API_KEY = "aNPobAtZIsrciuiwkj12ov"   # 建议填：OpenAlex api_key
MAILTO = "daisyfire0720@hotmail.com"

BASE = "https://api.openalex.org"

PER_PAGE = 200

# Throttling (tune if you hit rate limits)
SLEEP_LIST_CALL = 0.12
SLEEP_DETAIL_CALL = 0.05

TOP_N_BAR = 20
TOP_N_INSTITUTIONS_FOR_MAP = None  # e.g. 120; None means show all with geo


# =========================
# Session + Robust JSON GET
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (OpenAlexCitationMap; contact=mailto)"
})

def _get_json(url, params=None, max_retries=8, base_sleep=0.6):
    """
    Robust GET:
    - Retries on 429/5xx
    - Retries if response isn't JSON (HTML/empty)
    """
    if not url:
        raise ValueError("URL is empty/None")

    params = params.copy() if params else {}
    if API_KEY:
        params["api_key"] = API_KEY
    if MAILTO:
        params["mailto"] = MAILTO

    last_status = None
    last_head = None

    for i in range(max_retries):
        r = session.get(url, params=params, timeout=45)
        last_status = r.status_code
        text = r.text or ""
        last_head = text[:200].replace("\n", " ")

        if r.status_code == 200:
            ctype = (r.headers.get("Content-Type") or "").lower()
            if "json" in ctype and text.strip():
                try:
                    return r.json()
                except Exception:
                    pass  # retry
            # 200 but not json (html/empty) -> retry
        elif r.status_code in (429, 500, 502, 503, 504):
            pass
        else:
            raise RuntimeError(f"Request failed: {r.status_code}\nURL: {r.url}\nBody head: {last_head}")

        time.sleep((2 ** i) * base_sleep)

    raise RuntimeError(f"Failed after retries. Last status={last_status}\nURL: {url}\nBody head: {last_head}")


def to_api_work_url(openalex_work_id: str) -> str:
    """
    Convert OpenAlex work ID to API endpoint:
      https://openalex.org/Wxxxx -> https://api.openalex.org/works/Wxxxx
      Wxxxx -> https://api.openalex.org/works/Wxxxx
    """
    oid = (openalex_work_id or "").strip()
    if oid.startswith("https://openalex.org/"):
        oid = oid.replace("https://openalex.org/", "")
    if oid.startswith("http://openalex.org/"):
        oid = oid.replace("http://openalex.org/", "")
    return f"{BASE}/works/{oid}"


def to_api_inst_url(openalex_inst_id: str) -> str:
    """
    Convert OpenAlex institution ID to API endpoint.
    """
    iid = (openalex_inst_id or "").strip()
    if iid.startswith("https://openalex.org/"):
        iid = iid.replace("https://openalex.org/", "")
    if iid.startswith("http://openalex.org/"):
        iid = iid.replace("http://openalex.org/", "")
    return f"{BASE}/institutions/{iid}"


# =========================
# Step 1) Get target work
# =========================
target_work = _get_json(f"{BASE}/works/https://doi.org/{TARGET_DOI}")
target_openalex_id = target_work.get("id")
target_title = target_work.get("display_name")
target_year = target_work.get("publication_year")

print("Target title:", target_title)
print("Target year :", target_year)
print("Target OAID :", target_openalex_id)

if not target_openalex_id:
    raise RuntimeError("Failed to resolve target OpenAlex ID. Check DOI.")


# =========================
# Step 2) Fetch citing works (forward citations)
# Using cursor paging: /works?filter=cites:<target_id>
# =========================
citing_rows = []
cursor = "*"

while True:
    data = _get_json(
        f"{BASE}/works",
        params={
            "filter": f"cites:{target_openalex_id}",
            "per-page": PER_PAGE,
            "cursor": cursor,
            "select": "id,display_name,publication_year,doi,primary_location,cited_by_count"
        }
    )

    results = data.get("results", [])
    if not results:
        break

    for w in results:
        citing_rows.append({
            "citing_openalex_id": w.get("id"),
            "citing_title": w.get("display_name"),
            "citing_year": w.get("publication_year"),
            "citing_doi": (w.get("doi") or "").replace("https://doi.org/", ""),
            "citing_venue": (w.get("primary_location") or {}).get("source", {}).get("display_name"),
            "cited_by_count": w.get("cited_by_count"),
        })

    cursor = data.get("meta", {}).get("next_cursor")
    if not cursor:
        break

    time.sleep(SLEEP_LIST_CALL)

citing_df = (
    pd.DataFrame(citing_rows)
      .drop_duplicates(subset=["citing_openalex_id"])
      .reset_index(drop=True)
)

print("Total citing works:", len(citing_df))

# Save citing works list
citing_df.to_csv("citing_works.csv", index=False)


# =========================
# Step 3) Country distribution (paper-level unique countries)
# =========================
paper_to_countries = {}
paper_country_list = []

for idx, row in citing_df.iterrows():
    wid = row["citing_openalex_id"]
    if not wid:
        continue

    wobj = _get_json(
        to_api_work_url(wid),
        params={"select": "id,authorships"}  # nested select not allowed; take authorships and parse
    )

    countries = set()
    for auth in (wobj.get("authorships") or []):
        for inst in (auth.get("institutions") or []):
            cc = inst.get("country_code")
            if cc:
                countries.add(cc)

    cc_sorted = sorted(countries)
    paper_to_countries[wid] = cc_sorted

    for cc in cc_sorted:
        paper_country_list.append(cc)

    if (idx + 1) % 20 == 0:
        print(f"Processed countries for {idx+1}/{len(citing_df)} citing papers...")

    time.sleep(SLEEP_DETAIL_CALL)

citing_df["country_codes"] = citing_df["citing_openalex_id"].map(lambda x: ";".join(paper_to_countries.get(x, [])))

country_counts = Counter(paper_country_list)
country_df = (
    pd.DataFrame(country_counts.items(), columns=["country_code", "num_citing_papers"])
      .sort_values("num_citing_papers", ascending=False)
      .reset_index(drop=True)
)

print("\nTop countries:")
print(country_df.head(10))

country_df.to_csv("country_counts.csv", index=False)


# =========================
# Step 4) Gephi nodes/edges (optional but useful)
# =========================
edges_df = pd.DataFrame({
    "source": [target_openalex_id] * len(citing_df),
    "target": citing_df["citing_openalex_id"].tolist(),
    "type": ["Directed"] * len(citing_df)
})

nodes = [{"id": target_openalex_id, "label": target_title, "year": target_year, "country_codes": ""}]
for _, r in citing_df.iterrows():
    nodes.append({
        "id": r["citing_openalex_id"],
        "label": r["citing_title"],
        "year": r["citing_year"],
        "country_codes": r["country_codes"]
    })
nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=["id"]).reset_index(drop=True)

edges_df.to_csv("edges.csv", index=False)
nodes_df.to_csv("nodes.csv", index=False)


# =========================
# Step 5) Institution geo pin map data
# Count: per institution, number of citing papers (paper-level unique institutions)
# =========================
inst_to_papers = defaultdict(set)

for i, row in citing_df.iterrows():
    wid = row["citing_openalex_id"]
    if not wid:
        continue

    wobj = _get_json(
        to_api_work_url(wid),
        params={"select": "id,authorships"}
    )

    inst_ids = set()
    for auth in (wobj.get("authorships") or []):
        for inst in (auth.get("institutions") or []):
            inst_id = inst.get("id")
            if inst_id:
                inst_ids.add(inst_id)

    for inst_id in inst_ids:
        inst_to_papers[inst_id].add(wid)

    if (i + 1) % 10 == 0:
        print(f"Processed institutions for {i+1}/{len(citing_df)} citing papers...")

    time.sleep(SLEEP_DETAIL_CALL)

# Fetch institution geo
inst_rows = []
for j, (inst_id, paper_set) in enumerate(inst_to_papers.items(), start=1):
    inst_obj = _get_json(
        to_api_inst_url(inst_id),
        params={"select": "id,display_name,country_code,geo"}
    )
    geo = inst_obj.get("geo") or {}
    lat = geo.get("latitude")
    lon = geo.get("longitude")
    city = geo.get("city")

    if lat is None or lon is None:
        continue

    inst_rows.append({
        "institution_id": inst_obj.get("id"),
        "institution_name": inst_obj.get("display_name"),
        "country_code": inst_obj.get("country_code"),
        "city": city,
        "latitude": lat,
        "longitude": lon,
        "num_citing_papers": len(paper_set),
    })

    if j % 50 == 0:
        print(f"Fetched geo for {j}/{len(inst_to_papers)} institutions...")
    time.sleep(SLEEP_DETAIL_CALL)

inst_geo_df = (
    pd.DataFrame(inst_rows)
      .sort_values("num_citing_papers", ascending=False)
      .reset_index(drop=True)
)

if TOP_N_INSTITUTIONS_FOR_MAP is not None:
    inst_geo_df = inst_geo_df.head(TOP_N_INSTITUTIONS_FOR_MAP)

inst_geo_df.to_csv("institution_geo_counts.csv", index=False)
print("Saved: institution_geo_counts.csv")


# =========================
# Step 6) Plotting
# =========================

# ---- Helper: ISO2 -> ISO3 ----
def iso2_to_iso3(iso2: str):
    iso2 = (iso2 or "").strip().upper()
    if not iso2:
        return None
    special = {"XK": "XKX"}
    if iso2 in special:
        return special[iso2]
    c = pycountry.countries.get(alpha_2=iso2)
    return c.alpha_3 if c else None


# ---- (A) World choropleth (country-level) ----
plot_df = country_df.copy()
plot_df["country_code"] = plot_df["country_code"].astype(str).str.upper().str.strip()
plot_df["iso3"] = plot_df["country_code"].apply(iso2_to_iso3)
plot_df = plot_df.dropna(subset=["iso3"]).reset_index(drop=True)

map_fig = px.choropleth(
    plot_df,
    locations="iso3",
    color="num_citing_papers",
    hover_name="country_code",
    hover_data={"num_citing_papers": True, "iso3": False},
    labels={"num_citing_papers": "Number of Citing Papers"},
    title="Geographic Distribution of Citing Papers"
)
map_fig.update_layout(mapbox_style="open-street-map", margin=dict(l=20, r=20, t=70, b=20))
map_fig.write_html("citation_country_map.html")
print("Saved: citation_country_map.html")
try:
    map_fig.write_image("citation_country_map.png", scale=2)
    print("Saved: citation_country_map.png")
except Exception as e:
    print("PNG export skipped. Install kaleido: pip install -U kaleido")
    print("Error:", e)

map_fig.show()


# ---- (B) Country bar chart (Top N) ----
bar_df = plot_df.sort_values("num_citing_papers", ascending=False).head(TOP_N_BAR)

bar_fig = px.bar(
    bar_df,
    x="country_code",
    y="num_citing_papers",
    labels={"country_code": "Country", "num_citing_papers": "Number of Citing Papers"},
    title=f"Top {TOP_N_BAR} Countries by Number of Citing Papers"
)
bar_fig.update_layout(margin=dict(l=20, r=20, t=70, b=20), xaxis_tickangle=-45)
bar_fig.write_html("citation_country_bar.html")
print("Saved: citation_country_bar.html")
try:
    bar_fig.write_image("citation_country_bar.png", scale=2)
    print("Saved: citation_country_bar.png")
except Exception as e:
    print("PNG export skipped. Install kaleido: pip install -U kaleido")
    print("Error:", e)

bar_fig.show()


# ---- (C) Institution pin map (scatter_geo) ----
if not inst_geo_df.empty:
    pin_fig = px.scatter_geo(
        inst_geo_df,
        lat="latitude",
        lon="longitude",
        size="num_citing_papers",
        hover_name="institution_name",
        hover_data={"city": True, "country_code": True, "num_citing_papers": True},
        projection="natural earth",
        title="Institution-level Locations of Citing Authors"
    )
    pin_fig.update_layout(margin=dict(l=20, r=20, t=70, b=20))
    pin_fig.write_html("citation_institution_pin_map.html")
    print("Saved: citation_institution_pin_map.html")
    try:
        pin_fig.write_image("citation_institution_pin_map.png", scale=2)
        print("Saved: citation_institution_pin_map.png")
    except Exception as e:
        print("PNG export skipped. Install kaleido: pip install -U kaleido")
        print("Error:", e)

    pin_fig.show()
else:
    print("No institution geo data available (inst_geo_df is empty).")


Target title: Equity issues associated with U.S. plug-in electric vehicle income tax credits
Target year : 2021
Target OAID : https://openalex.org/W4200060457
Total citing works: 24
Processed countries for 20/24 citing papers...

Top countries:
  country_code  num_citing_papers
0           US                 12
1           CN                  3
2           AU                  2
3           FI                  1
4           IN                  1
5           BE                  1
6           NL                  1
7           CZ                  1
8           PL                  1
9           IE                  1
Processed institutions for 10/24 citing papers...
Processed institutions for 20/24 citing papers...
Saved: institution_geo_counts.csv
Saved: citation_country_map.html
Saved: citation_country_map.png


Saved: citation_country_bar.html
Saved: citation_country_bar.png


Saved: citation_institution_pin_map.html
Saved: citation_institution_pin_map.png


In [5]:
# ---- (C) Institution pin map (scatter_mapbox) ----
if not inst_geo_df.empty:
    pin_fig = px.scatter_map(
        inst_geo_df,
        lat="latitude",
        lon="longitude",
        size="num_citing_papers",
        hover_name="institution_name",
        hover_data={"city": True, "country_code": True, "num_citing_papers": True, "latitude": False, "longitude": False},
        title="Institution-level Locations of Citing Authors (OpenAlex)",
        size_max=50,
        zoom=1
    )
    
    pin_fig.update_traces(marker=dict(symbol="marker", opacity=0.8, size=10))
    pin_fig.update_layout(
        mapbox_style="open-street-map",
        margin=dict(l=20, r=20, t=70, b=20)
    )
    pin_fig.write_html("citation_institution_pin_map.html")
    print("Saved: citation_institution_pin_map.html")
    try:
        pin_fig.write_image("citation_institution_pin_map.png", scale=2)
        print("Saved: citation_institution_pin_map.png")
    except Exception as e:
        print("PNG export skipped. Install kaleido: pip install -U kaleido")
        print("Error:", e)

    pin_fig.show()
else:
    print("No institution geo data available (inst_geo_df is empty).")

Saved: citation_institution_pin_map.html
Saved: citation_institution_pin_map.png
